In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
!pip install llama-index llama-index-llms-openai llama-index-embeddings-openai

In [ ]:
!pip install llama-index llama-index-llms-huggingface llama-index-embeddings-huggingface transformers accelerate bitsandbytes

In [ ]:
!pip install llama-index-readers-file

**for large data on hugging face**

In [ ]:
from datasets import load_dataset
import os
import re

# 1. Load the dataset
print("Downloading large movie dataset...")
dataset = load_dataset("AIatMongoDB/embedded_movies", split="train")

# 2. Setup your movie_data folder
output_dir = "./movie_data_large"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 3. Save as individual files (NTCIR Style)
print("Processing and saving files...")
for i, item in enumerate(dataset):
    # Skip records with missing essential data
    if not item.get('fullplot') or not item.get('title'):
        continue

    title = item['title']
    plot = item['fullplot']

    # FIX: Use 'or []' to handle None values gracefully
    directors = item.get('directors') or []
    cast = item.get('cast') or []

    # Sanitize title for filename
    safe_title = re.sub(r'[^\w\s]', '', title).strip().replace(' ', '_')
    filename = f"{output_dir}/{safe_title}_{i}.txt"

    try:
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"MOVIE TITLE: {title}\n")
            f.write(f"DIRECTORS: {', '.join(directors)}\n")
            f.write(f"CAST: {', '.join(cast)}\n")
            f.write(f"STORY PLOT: {plot}")
    except Exception as e:
        print(f"Skipping {title} due to error: {e}")

print(f"Success! Your library now contains {len(os.listdir(output_dir))} movie files.")

# Learning step 4:

In [ ]:
!pip install ragatouille rank_bm25 pinecone-client langchain-pinecone langchain-huggingface langchain-community langgraph groq

In [ ]:
# 1. Uninstall the conflicting versions
!pip uninstall -y langchain langchain-community ragatouille
!pip install "transformers<4.40.0" "tokenizers<0.19.0"
# 2. Install compatible "Step 4" stack
!pip install "langchain>=0.2.0" "langchain-community>=0.2.0" ragatouille

In [ ]:
!pip install langchain-classic

In [ ]:
!pip install --upgrade huggingface_hub transformers

In [ ]:
!pip install "huggingface_hub>=0.20.0" "transformers>=4.39.0" "ragatouille==0.0.9post2"

In [ ]:
# 1. Update the core Hugging Face utilities
!pip install --upgrade huggingface_hub transformers

# 2. Re-install the specific ragatouille version to ensure it doesn't break
!pip install ragatouille==0.0.9post2

In [ ]:
# Force-reinstall core dependencies to ensure they are synchronized
!pip install --force-reinstall huggingface_hub==0.28.0 transformers==4.40.0 tokenizers==0.19.0
!pip install ragatouille==0.0.9post2

In [ ]:
# 1. Wipe the broken installations completely
!pip uninstall -y transformers tokenizers huggingface_hub ragatouille colbert-ai


!pip install transformers==4.40.0 tokenizers==0.19.0 huggingface_hub==0.28.0
!pip install ragatouille==0.0.9post2



In [ ]:
import sys
import torch
from types import ModuleType

# 1. LangChain Bridge
for mod in ["langchain.retrievers", "langchain.retrievers.document_compressors", "langchain.retrievers.document_compressors.base"]:
    if mod not in sys.modules:
        sys.modules[mod] = ModuleType(mod)
import langchain_core.documents
sys.modules["langchain.retrievers.document_compressors.base"].BaseDocumentCompressor = langchain_core.documents.BaseDocumentCompressor

# 2. THE SURGICAL PATCH

torch.nn.Module.all_tied_weights_keys = {}

print("✅ Surgical Torch patch applied (Attribute mode)")

# 3. Load RAGatouille
from ragatouille import RAGPretrainedModel

try:
    RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")
    print("\n🚀 SUCCESS: ColBERT is fully loaded and ready!")
except Exception as e:
    print(f"❌ Snag: {e}")

In [ ]:
import os, time, re
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

os.environ["PINECONE_API_KEY"] = "pinecone_api_key_here"
index_name = "movie-index"

def extract_deep_metadata(text, filename):
    text_sample = text[:2000].lower()
    meta = {
        "title": os.path.basename(filename).replace('.txt', ''),
        "year": 0,
        "genre": "Unknown",
        "setting_period": "Modern",
        "tone": "Neutral",
        "archetype": "General",
        "plot_element": "Standard",
        "country": "Unknown",
        "budget_scale": "Medium",
        "language": "English"
    }

    # 1. Year & Period
    year_match = re.search(r'(19|20)\d{2}', text_sample)
    if year_match:
        meta["year"] = int(year_match.group())
        if meta["year"] < 1950: meta["setting_period"] = "Vintage/Classic"
        elif meta["year"] < 1980: meta["setting_period"] = "Retro"

    # 2. Genre & Tone Logic
    mappings = {
        "genre": {"noir": "Neo-Noir", "thriller": "Thriller", "comedy": "Comedy", "horror": "Horror", "crime": "Crime"},
        "tone": {"dark": "Dark", "gritty": "Gritty", "melancholic": "Melancholic", "hopeful": "Hopeful"},
        "archetype": {"informer": "Informer", "revolutionary": "Revolutionary", "thief": "Protagonist-Thief", "detective": "Detective"},
        "plot_element": {"heist": "Heist", "romance": "Romance", "betrayal": "Betrayal", "rain": "Weather-Atmospheric"}
    }

    for key, categories in mappings.items():
        for keyword, value in categories.items():
            if keyword in text_sample:
                meta[key] = value
                break
    return meta

# Execution: Load, Split, and Upload
loader = DirectoryLoader('./movie_data_large', glob="./*.txt", loader_cls=TextLoader)
raw_docs = loader.load()

for doc in raw_docs:
    doc.metadata.update(extract_deep_metadata(doc.page_content, doc.metadata['source']))

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(raw_docs)

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = PineconeVectorStore.from_documents(docs, embeddings, index_name=index_name)
print("✅ Deep Metadata Index Built.")

In [ ]:
import json
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
from groq import Groq
from langchain_community.retrievers import BM25Retriever
from ragatouille import RAGPretrainedModel

# 1. Initialize ColBERT (Late Interaction Model)
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

# 2. Initialize BM25 on the full document set
bm25_retriever = BM25Retriever.from_documents(docs)

client = Groq(api_key="grok_api_key_here")

class GraphState(TypedDict):
    query: str
    sub_queries: List[str]
    current_step: int
    cumulative_ans: str
    context: str
    prediction: str
    relevance_score: str
    iterations: int
    filters: Dict[str, Any]

# --- THE ELITE HYBRID SEARCH UTILITY ---
def elite_hybrid_search(query: str, filters: dict, k: int = 2):
    """BM25 Union -> Metadata Filter -> ColBERT Rerank"""
    print(f"🔍 Deep Searching for: {query}")


    initial_candidates = bm25_retriever.invoke(query, k=10)


    filtered_candidates = []
    if filters and "year" in filters:
        y_min, y_max = filters["year"]["$gte"], filters["year"]["$lte"]
        for d in initial_candidates:
            if y_min <= d.metadata.get("year", 0) <= y_max:
                filtered_candidates.append(d)
    else:
        filtered_candidates = initial_candidates

    if not filtered_candidates:
        return []

    texts = [d.page_content for d in filtered_candidates]

    reranked_results = RAG.rerank(query=query, documents=texts, k=k)

    final_docs = []
    for res in reranked_results:
        original_doc = next(d for d in filtered_candidates if d.page_content == res['content'])
        final_docs.append(original_doc)

    return final_docs

# --- LANGGRAPH NODES ---

def decompose_node(state: GraphState):
    prompt = f"Analyze query: '{state['query']}'. Extract JSON: year_start, year_end (±2y), genre, tone, setting. JSON ONLY."
    res = client.chat.completions.create(model="llama-3.3-70b-versatile", messages=[{"role":"user","content":prompt}], response_format={"type":"json_object"})
    meta = json.loads(res.choices[0].message.content)

    filters = {}
    if meta.get('year_start'): filters["year"] = {"$gte": meta['year_start'], "$lte": meta['year_end']}
    if meta.get('genre') and meta['genre'] != "None": filters["genre"] = meta['genre']

    return {"filters": filters, "sub_queries": [f"Focus on {meta.get('tone','plot')} and {meta.get('setting','atmosphere')}"], "current_step": 0, "iterations": 0}

def research_node(state: GraphState):
    q = state["sub_queries"][state["current_step"]]
    docs = elite_hybrid_search(q, state["filters"], k=2)

    new_context = "\n".join([f"[{d.metadata['title']}]: {d.page_content}" for d in docs])
    return {"cumulative_ans": state["cumulative_ans"] + "\n" + new_context, "current_step": state["current_step"] + 1}

def final_retrieve_node(state: GraphState):
    docs = elite_hybrid_search(state["query"], state["filters"], k=1)
    return {"context": "\n".join([d.page_content for d in docs])}

def generate_node(state: GraphState):
    print("\n--- 📝 GENERATING NTCIR-19 OFFICIAL SUBMISSION ---")

    prompt = f"""
    You are an NTCIR-19 Researcher submitting to the R2C2 Task.

    USER QUERY: {state['query']}
    RETRIEVED CONTEXT: {state['context']}

    INSTRUCTIONS:
    - Topic ID: Generate a unique ID (e.g., TOT-2026-001).
    - Doc ID: The ID or title of the movie.
    - Confidence: A score between 0 and 100 (NTCIR uses 0-100 scale for AC runs).
    - Nuggets: Factual claims extracted from the context that prove the answer.
    - Modesty: Why is the confidence not 100? (e.g. "Year match but plot details are generic").

    OUTPUT JSON ONLY:
    {{
      "topic_id": "TOT-2026-001",
      "doc_id": "1942_A_LOVE_STORY",
      "confidence": 94,
      "answer": "1942: A Love Story (1994)",
      "nuggets": ["Set in 1942 during British Raj", "Manisha Koirala is freedom fighter daughter", "Anil Kapoor is British informer son"],
      "modesty": "High confidence due to specific character dynamics and year match."
    }}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    data = json.loads(response.choices[0].message.content)

    ntcir_table = f"""
| NTCIR-19 Field | System Value |
| :--- | :--- |
| **Topic ID** | {data['topic_id']} |
| **Doc ID** | {data['doc_id']} |
| **Confidence** | {data['confidence']}/100 |
| **Answer** | {data['answer']} |
| **Evidence Nuggets** | {", ".join(data['nuggets'])} |
| **Modesty Note** | {data['modesty']} |
    """

    return {"prediction": ntcir_table}
# --- CONTINUATION: GRADING, REWRITING, AND GRAPH ASSEMBLY ---

def grade_node(state: GraphState):
    """
    Determines if the ColBERT-retrieved context is sufficient.
    """
    print("\n--- ⚖️ GRADING RELEVANCE (COLBERT-VERIFIED) ---")

    if not state["context"].strip():
        return {"relevance_score": "no"}

    prompt = f"""
    You are a strict Research Grader.
    User Query: {state['query']}
    Retrieved Context: {state['context']}

    Does the retrieved context provide a direct match or a very close thematic match?
    (e.g., if the user asked for a 1979 rainstorm heist and you found a 1981 rainstorm heist, that is a YES).

    Answer ONLY 'YES' or 'NO'.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )

    relevance = response.choices[0].message.content.strip().lower()
    relevance_score = "yes" if "yes" in relevance else "no"

    return {"relevance_score": relevance_score}

def agentic_rewrite_node(state: GraphState):
    """
    Optimizes the search query if the previous one failed to find a match.
    """
    print("\n--- 🔄 RE-PLANNING: OPTIMIZING KEYWORDS ---")

    prompt = f"""
    The previous search strategy failed to find a match for: "{state['query']}"
    History of attempts: {state['cumulative_ans']}

    Generate a new, highly specific search query focusing on technical nouns and specific setting details.
    Example: "vault thermal lance rainstorm"
    Output only the new search query.
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
    )

    new_q = response.choices[0].message.content.strip()

    return {
        "sub_queries": [new_q],
        "current_step": 0,
        "iterations": state.get("iterations", 0) + 1
    }

# --- 5. GRAPH LOGIC (ROUTING) ---

def decide_after_grading(state: GraphState):
    """
    Decision logic: Should we generate the answer or try searching again?
    """
    if state["relevance_score"] == "yes":
        return "generate"

    # If we've reached the 'Rewrite Budget' (e.g., 2 attempts), we generate anyway
    # to avoid infinite loops, but the LLM will explain the lack of data.
    if state["iterations"] >= 2:
        print("⚠️ Rewrite budget exhausted. Proceeding to final report.")
        return "generate"

    # Otherwise, try to rewrite the query.
    return "research_more"

# --- 6. WORKFLOW ASSEMBLY ---

workflow = StateGraph(GraphState)

# Add all nodes
workflow.add_node("decompose", decompose_node)
workflow.add_node("research", research_node)
workflow.add_node("final_retrieve", final_retrieve_node)
workflow.add_node("grade", grade_node)
workflow.add_node("agentic_rewrite", agentic_rewrite_node)
workflow.add_node("generate", generate_node)

# Define edges
workflow.set_entry_point("decompose")
workflow.add_edge("decompose", "research")
workflow.add_edge("research", "final_retrieve")
workflow.add_edge("final_retrieve", "grade")

# Conditional Logic
workflow.add_conditional_edges(
    "grade",
    decide_after_grading,
    {
        "generate": "generate",
        "research_more": "agentic_rewrite"
    }
)

# Loop back from rewrite to research
workflow.add_edge("agentic_rewrite", "research")

# End the graph
workflow.add_edge("generate", END)

# Compile the application
app = workflow.compile()

print("🚀 Step 4 LangGraph (Hybrid BM25 + ColBERT) is ready.")

In [ ]:
# Define the input state
inputs = {
    "query": "I'm looking for a movie from the early 90s (around 1994). It's a romance set during the British Raj in India. The plot involves a revolutionary's daughter and the son of a British informer. I remember a very famous scene in the rain with a melodic song.",
    "iterations": 0,
    "cumulative_ans": "",
    "current_step": 0
}

# Run the Graph
print("🎬 Starting NTCIR-19 Retrieval Loop...")
final_state = app.invoke(inputs)

# Display the NTCIR-19 Formatted Output
print(final_state["prediction"])

In [ ]:
import gradio as gr

# 1. Define the Theme
theme = gr.themes.Default(
    primary_hue="orange",
    secondary_hue="amber",
    neutral_hue="slate",
).set(
    body_background_fill="#1a1a1a",
    block_background_fill="#2d2d2d",
    body_text_color="#ffffff",
    block_title_text_color="#ffa500",
    block_label_text_color="#ffa500",
    button_primary_background_fill="#ff8c00",
    button_primary_text_color="#ffffff",
    input_background_fill="#3d3d3d",
    input_border_color="#ff8c00",
)

# 2. Custom CSS (Guarantees White Text and table styling)
custom_css = """
footer {display: none !important;}
.gradio-container {border: 2px solid #ff8c00 !important; border-radius: 15px;}
textarea, input {color: white !important;}
#ntcir-table {background-color: #262626; padding: 15px; border-radius: 10px; border: 1px solid #ff8c00;}
"""

def run_agentic_search(user_query):
    try:
        # SYNCED WITH GRAPHSTATE: Initialize all required keys
        inputs = {
            "query": user_query,
            "filters": {},
            "context": "",
            "prediction": "",
            "log": "🔄 Initializing hybrid search..."
        }

        # Execute the LangGraph 'app'
        result_state = app.invoke(inputs)

        # Extract the markdown table (prediction) and the thinking trace (log)
        final_report = result_state.get("prediction", "### ❌ Error\nNo report generated.")
        thoughts = result_state.get("log", "Trace empty.")

        return final_report, thoughts

    except Exception as e:
        return f"### ❌ System Error\n{str(e)}", f"Error details: {e}"

# 3. Build the UI
with gr.Blocks(title="NTCIR-19 Researcher") as demo:
    gr.HTML("<h1 style='color:#ff8c00; text-align:center;'>🎬 NTCIR-19 AGENTIC MOVIE SEARCH</h1>")
    gr.Markdown("<p style='text-align:center; color:white;'>Learning Step 4: Hybrid BM25 + BGE Reranking</p>")

    with gr.Row():
        with gr.Column(scale=1):
            query_box = gr.Textbox(
                label="Search Query",
                placeholder="Search by plot, vibe, or specific year (e.g., 1950s wrestling noir)...",
                lines=4
            )
            search_btn = gr.Button("🔍 START AGENTIC LOOP", variant="primary")

            with gr.Accordion("🧠 Thinking Trace (Hybrid Logic)", open=True):
                trace_box = gr.Markdown("*Agent logs will appear here during search...*")

        with gr.Column(scale=1):
            gr.Markdown("### 🏆 NTCIR-19 Submission Result")
            output_table = gr.Markdown(elem_id="ntcir-table", value="*Waiting for results...*")

    # Connect the button
    search_btn.click(
        fn=run_agentic_search,
        inputs=query_box,
        outputs=[output_table, trace_box]
    )

# 4. Launch (Correct way for Gradio 6.0 and Colab/Spaces)
if __name__ == "__main__":
    demo.launch(
        theme=theme,
        css=custom_css,
        share=True,
        debug=True
    )

In [ ]:
import gradio as gr

# 1. Define the High-Contrast Orange Theme
# We use 'orange' as the primary and 'amber' as the secondary
theme = gr.themes.Default(
    primary_hue="orange",
    secondary_hue="amber",
    neutral_hue="slate",
).set(
    # Core Backgrounds
    body_background_fill="#1a1a1a",       # Deep Charcoal
    block_background_fill="#2d2d2d",      # Dark Grey-Orange

    # Text Colors
    body_text_color="#ffffff",            # Global White Text
    block_title_text_color="#ffa500",     # Orange Headers
    block_label_text_color="#ffa500",     # Orange Labels

    # Button Styling
    button_primary_background_fill="#ff8c00", # Pure Orange
    button_primary_background_fill_hover="#e67e00",
    button_primary_text_color="#ffffff",

    # Input Styling
    input_background_fill="#3d3d3d",
    input_border_color="#ff8c00",
)

# 2. Custom CSS for extra "pop" (Guarantees White Text in Inputs)
custom_css = """
footer {display: none !important;}
.gradio-container {border: 2px solid #ff8c00 !important; border-radius: 15px;}
textarea, input {color: white !important;}
#ntcir-table {background-color: #262626; padding: 10px; border-radius: 8px;}
"""

def run_agentic_search(user_query):
    # This connects to your LangGraph 'app'
    inputs = {
        "query": user_query,
        "iterations": 0,
        "cumulative_ans": "",
        "current_step": 0
    }

    # Execute the graph
    result_state = app.invoke(inputs)

    # Extract the markdown table and the thinking trace
    final_report = result_state.get("prediction", "No result generated.")
    thoughts = result_state.get("cumulative_ans", "Trace empty.")

    return final_report, thoughts

# 3. Build the UI
with gr.Blocks(theme=theme, css=custom_css, title="NTCIR-19 Researcher") as demo:
    gr.HTML("<h1 style='color:#ff8c00; text-align:center;'>🎬 NTCIR-19 AGENTIC MOVIE SEARCH</h1>")
    gr.Markdown("<p style='text-align:center; color:white;'>Learning Step 4: Hybrid BM25 + ColBERT Reranking</p>")

    with gr.Row():
        with gr.Column(scale=1):
            query_box = gr.Textbox(
                label="Step 4: Search Query",
                placeholder="Search by plot, vibe, or specific year (e.g., 1942: A Love Story)...",
                lines=4
            )
            search_btn = gr.Button("🔍 START AGENTIC LOOP", variant="primary")

            with gr.Accordion("🧠 Thinking Trace (ColBERT Interactions)", open=True):
                trace_box = gr.Markdown("*Agent logs will appear here during search...*")

        with gr.Column(scale=1):
            gr.Markdown("### 🏆 NTCIR-19 Submission Result")
            # We use an elem_id here so our CSS can target it if needed
            output_table = gr.Markdown(elem_id="ntcir-table", value="*Waiting for results...*")

    # Connect the button
    search_btn.click(
        fn=run_agentic_search,
        inputs=query_box,
        outputs=[output_table, trace_box]
    )

# 4. Launch with a public link
if __name__ == "__main__":
    demo.launch(debug=True)

In [ ]:
[
  {
    "query": "noir movie london wrestling",
    "relevant_movies": ["Night and the City"]
  },
  {
    "query": "movie with black monolith and space journey",
    "relevant_movies": ["2001: A Space Odyssey"]
  },
  {
    "query": "crime thriller coffee shop scene cop and thief",
    "relevant_movies": ["Heat"]
  },
  {
    "query": "movie about dream invasion and layered reality",
    "relevant_movies": ["Inception"]
  },
  {
    "query": "ship sinking love story set on titanic",
    "relevant_movies": ["Titanic"]
  },
  {
    "query": "movie where a computer becomes dangerous AI HAL",
    "relevant_movies": ["2001: A Space Odyssey"]
  },
  {
    "query": "batman movie with joker chaos and dark tone",
    "relevant_movies": ["The Dark Knight"]
  },
  {
    "query": "movie about dinosaurs created using DNA in a park",
    "relevant_movies": ["Jurassic Park"]
  },
  {
    "query": "space movie where astronauts try to survive on another planet",
    "relevant_movies": ["The Martian"]
  },
  {
    "query": "movie about time travel and paradoxes with multiple timelines",
    "relevant_movies": ["Interstellar"]
  }
]